# Census MTIS 制造业库存销售比变化：H63 现金信息来源门

完整Stage1 f696+b472父方案及SOXX最终硬门不变。冻结比率变化、H63现金价值方向和支持门后，先仅检查103个月原件映射、标签成熟时点、明确晚修改的掩码与取值多样性。来源门失败必须在读取SOXX收益、拟合和账户前停下。

In [1]:
import hashlib,json,os,pathlib,shlex,sys
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_census_mtis_h63_cash_necessary_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
parent=root/src['parent_card'];traincard=root/src['training_source_card'];heldcard=root/src['held_source_card'];p_art=parent/'qlib_recorder'/src['parent_experiment_id']/src['parent_recorder_id']/'artifacts';t_art=traincard/'qlib_recorder'/src['training_source_experiment_id']/src['training_source_recorder_id']/'artifacts';h_art=heldcard/'qlib_recorder'/src['held_source_experiment_id']/src['held_source_recorder_id']/'artifacts'
for p,h in [(parent/'summary.json',src['parent_summary_sha256']),(p_art/'inputs/features',src['parent_features_sha256']),(p_art/'inputs/daily_frame',src['parent_daily_frame_sha256']),(p_art/'inputs/timing',src['parent_timing_sha256']),(traincard/'summary.json',src['training_source_summary_sha256']),(t_art/'analysis/monthly_source',src['training_source_frame_sha256']),(heldcard/'summary.json',src['held_source_summary_sha256']),(h_art/'analysis/monthly_source',src['held_source_frame_sha256'])]:assert sha(p)==h,(p,sha(p),h)
assert json.loads((traincard/'summary.json').read_text())['NAS_readback']['verified'] and json.loads((heldcard/'summary.json').read_text())['NAS_readback']['verified']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider_uri']),region='us',kernels=1);cal=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert len(cal)==3966
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['parent_experiment_id'],recorder_id=src['parent_recorder_id']);assert rec.status=='FINISHED';timing=rec.load_object('inputs/timing')
with R.uri_context(str(traincard/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['training_source_experiment_id'],recorder_id=src['training_source_recorder_id']);assert rec.status=='FINISHED';training=rec.load_object('analysis/monthly_source')
with R.uri_context(str(heldcard/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['held_source_experiment_id'],recorder_id=src['held_source_recorder_id']);assert rec.status=='FINISHED';held_source=rec.load_object('analysis/monthly_source')
assert timing.index.equals(cal) and len(training)==60 and len(held_source)==43
source=pd.concat([training,held_source],ignore_index=True);assert len(source)==103 and source.economic_month.is_unique and source.status.eq('current_original_source_semantics_and_clock_qualified').all();source['event_trade_date']=pd.to_datetime(source.first_Qlib_session_strictly_after_release);source['release_date']=pd.to_datetime(source.release_date);source['inventory_ratio_delta']=source.manufacturers_adjusted_current_preliminary_ratio-source.manufacturers_adjusted_previous_revised_ratio;assert np.isfinite(source.inventory_ratio_delta.to_numpy(float)).all()
bridge=pd.DataFrame({'signal_date':timing.index,'event_trade_date':pd.DatetimeIndex(pd.to_datetime(timing.decision_at).dt.tz_localize(None).dt.normalize())});events=source.merge(bridge,on='event_trade_date',how='left',validate='one_to_one');assert events.signal_date.notna().all() and events.signal_date.is_unique and (events.release_date<events.event_trade_date).all()
maturity=pd.to_datetime(timing.return_available_at.shift(-62),utc=True);events['label_maturity_at']=maturity.reindex(events.signal_date).to_numpy();events['definitely_late_current_PDF']=events.server_Last_Modified_after_release_date.eq(True).to_numpy(bool)
train=events.loc[events.signal_date.between('2018-01-01','2022-12-31') & pd.to_datetime(events.label_maturity_at,utc=True).lt(pd.Timestamp('2023-01-03',tz='UTC'))].copy();held=events.loc[events.signal_date.between('2023-01-03','2026-10-07') & pd.to_datetime(events.label_maturity_at,utc=True).lt(pd.Timestamp('2026-10-10',tz='UTC')) & ~events.definitely_late_current_PDF].copy();train=train.sort_values('signal_date');held=held.sort_values('signal_date');coh=plan['preflight'];years={int(k):int(v) for k,v in held.groupby(held.signal_date.dt.year).size().to_dict().items()};assert len(train)==coh['mature_train_rows']==55 and len(held)==coh['mature_held_rows_after_late_vintage_mask']==41 and years=={2023:12,2024:12,2025:11,2026:6}
train_unique=int(train.inventory_ratio_delta.nunique());held_unique=int(held.inventory_ratio_delta.nunique());passed=train_unique>=coh['minimum_train_unique_delta_values'] and held_unique>=coh['minimum_held_unique_delta_values'];assert not passed and train_unique==13 and held_unique==5
preflight={'status':'source_event_and_H63_maturity_preflight_failed_held_numeric_variation','plan_sha256':sha(card/'plan.json'),'all_source_events':len(events),'train_mature_event_rows':len(train),'held_mature_event_rows':len(held),'held_year_event_rows':{str(k):v for k,v in years.items()},'train_unique_delta_values':train_unique,'held_unique_delta_values':held_unique,'minimum_held_unique_delta_values':coh['minimum_held_unique_delta_values'],'train_zero_delta_events':int(train.inventory_ratio_delta.eq(0).sum()),'held_zero_delta_events':int(held.inventory_ratio_delta.eq(0).sum()),'definitely_late_current_PDF_events_masked':int(events.definitely_late_current_PDF.sum()),'source_support_pass':False,'SOXX_outcome_values_read':0,'DatasetH_builds':0,'model_fits':0,'native_accounts':0};(card/'preflight.json').write_text(json.dumps(preflight,indent=2)+'\n')
result={'decision':'closed_fixed_MTiS_delta_H63_cash_preflight_low_held_variation','plan_sha256':preflight['plan_sha256'],'preflight':preflight,'rank_screen_executed':False,'bootstrap_executed':False,'historical_initial_public_bytes_qualified':False,'held_history_independent':False,'SOXX_outcome_values_read':0,'model_fits':0,'native_accounts':0,'next_action':'Do not fit or vary delta formula, sign, source years or H63 horizon on reused history. A new materially distinct source mechanism requires its own frozen proposal; full Stage1 0bp SOXX objective remains active.'};(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
state={'status':'finished_source_preflight_stop','plan_sha256':preflight['plan_sha256'],'D_calendar_calls':1,'parent_Recorder_object_loads':1,'source_Recorder_object_loads':2,'DatasetH_builds':0,'bootstrap_runs':0,'SOXX_outcome_values_read':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight);rec.save_objects(artifact_path='analysis',source_events=events.drop(columns=['release_date'],errors='ignore'),result=result);rec.log_metrics(train_events=len(train),held_events=len(held),held_unique_delta_values=held_unique,source_support_pass=0,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state.update(recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');ev=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(ev['nas_path'])).split()[0]==ev['sha256']
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'preflight':'preflight.json','result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':ev,'NAS_readback':{'sha256':ev['sha256'],'remote_hash_verified':True},'source_support_pass':False,'SOXX_outcome_values_read':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/source_events','analysis/result']},'limitations':['Only source identities, event timing and label maturity metadata were read; no H63 SOXX outcome values or rank were computed.','Current original PDF first-public bytes remain unqualified; held history is reused development data.','No Stage1 incremental receiver, full 0bp Tiered account or SOXX success.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(json.dumps({'decision':result['decision'],'train_events':len(train),'held_events':len(held),'train_unique':train_unique,'held_unique':held_unique,'recorder_id':rid}))


2026/10/23 15:47:39 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[20850:MainThread](2026-10-23 15:47:39,884) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[20850:MainThread](2026-10-23 15:47:39,886) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[20850:MainThread](2026-10-23 15:47:39,886) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[20850:MainThread](2026-10-23 15:47:39,991) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_census_mtis_h63_cash_necessary_information_20261023.


[20850:MainThread](2026-10-23 15:47:39,993) INFO - qlib.workflow - [exp.py:258] - Experiment 750803526457455101 starts running ...


[20850:MainThread](2026-10-23 15:47:40,072) INFO - qlib.workflow - [recorder.py:345] - Recorder 1b29285e844c4af195d1319e4c9cceb8 starts running under Experiment 750803526457455101 ...


[20850:MainThread](2026-10-23 15:47:41,239) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "closed_fixed_MTiS_delta_H63_cash_preflight_low_held_variation", "train_events": 55, "held_events": 41, "train_unique": 13, "held_unique": 5, "recorder_id": "1b29285e844c4af195d1319e4c9cceb8"}


# 结论：Census MTIS 比率月变化对 Stage1 H63 现金任务的来源门

冻结完整 Stage1 f696+b472 为后续经济父方案，并固定“同一原稿中的制造业合计库存销售比当期初值减上期修订值”这一列。103个月当前官方原稿映射到对应09:46决策之后的Qlib交易日；成熟训练事件 **55** 个，复用留出事件 **41** 个。明确晚修改的2026年四份当前PDF已掩码，未使用其数值评分。

在读取任何SOXX未来收益之前，留出事件的比率变化只有 **5 个按报告精度的不同取值**（−0.02、−0.01、0、+0.01、+0.02），其中16个为零，未达预注册的至少8种。训练事件的二进制浮点表面上有13种，按原稿两位小数精度合并后实际为10种；这不改变留出门失败。原生Qlib Recorder `1b29285e844c4af195d1319e4c9cceb8` 的103行来源表及NAS选定工件已冷恢复，精度复核见 `precision_audit.json`。

**决定：关闭该固定“月变化→H63现金价值”版本的来源支持门。** 没有读取SOXX H63结果、建立 DatasetH、计算相关或置信区间、拟合模型或运行账户。它不证明所有库存数据无效；不在复用市场历史上改成比率水平、改预测期或降低取值门来重开本版。另有历史初版字节未证实和全美制造业合计非半导体专属的来源限制。SOXX 0bp、2/5bp及独立前向门均未通过。
